# 05 - Neural Network Comparison and Explainability

This notebook adds one small feedforward neural network to answer a focused research question:

> Does a less directly interpretable nonlinear model improve failure prediction enough to justify its additional complexity and post-hoc explanation requirements?

The goal is comparison, not an assumption that deep learning must be better. The network uses the same safe features, held-out test records, and class-imbalance controls as the decision-tree experiment.

## 1. Neural Network Theory

After preprocessing, one machine record is represented by a vector $x$. The first hidden layer computes:

$$h_1=\operatorname{ReLU}(W_1x+b_1)$$

The second hidden layer computes:

$$h_2=\operatorname{ReLU}(W_2h_1+b_2)$$

The output is converted into failure probability:

$$\hat p=\sigma(W_3h_2+b_3)$$

The network learns weights through backpropagation and Adam optimization. ReLU permits nonlinear relationships; the sigmoid output remains between zero and one.

In [1]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import shap
from IPython.display import display
from lime.lime_tabular import LimeTabularExplainer
from sklearn.compose import ColumnTransformer
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.utils.class_weight import compute_sample_weight


def find_project_root(start_path):
    for candidate in [start_path, *start_path.parents]:
        if (candidate / "src" / "predictive_maintenance").exists():
            return candidate
    raise FileNotFoundError("Could not locate the project root.")


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from predictive_maintenance.classification import FailureClassificationExperiment
from predictive_maintenance.config import (
    CATEGORICAL_FEATURES,
    INPUT_FEATURES,
    NUMERIC_FEATURES,
    RANDOM_STATE,
)
from predictive_maintenance.data_loader import load_ai4i_dataset
from predictive_maintenance.neural_network import NeuralNetworkExperiment

OUTPUT_DIR = PROJECT_ROOT / "outputs" / "neural_network_comparison"
FIGURE_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", lambda value: f"{value:.4f}")
print(f"Project root: {PROJECT_ROOT}")

Project root: /Users/bijaypandey/Documents/Uni of Regina/ENSE_861/responsible_ai_project


/Users/bijaypandey/Documents/Uni of Regina/ENSE_861/responsible_ai_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. Load Data and Train the Fixed Architecture

The architecture is intentionally small: 8 transformed inputs, hidden layers of 16 and 8 neurons, and one binary output. With biases, it has only 289 trainable parameters.

The numeric features are standardized and `Type` is one-hot encoded. Scaling is important because gradient-based optimization is sensitive to feature magnitude.

In [ ]:
data = load_ai4i_dataset()
neural_experiment = NeuralNetworkExperiment(
    hidden_layer_sizes=(16, 8),
    alpha=0.001,
    learning_rate_init=0.001,
    max_iter=500,
    random_state=RANDOM_STATE,
).fit(data)

network = neural_experiment.model.named_steps["classifier"]
layer_sizes = [8, *network.hidden_layer_sizes, 1]
parameter_count = sum(
    layer_sizes[index] * layer_sizes[index + 1] + layer_sizes[index + 1]
    for index in range(len(layer_sizes) - 1)
)

print(f"Network layers: {layer_sizes}")
print(f"Trainable parameters: {parameter_count}")
print(f"Training iterations: {network.n_iter_}")
print(f"Best internal validation accuracy: {network.best_validation_score_:.4f}")

## 3. Class Imbalance, Loss, and Early Stopping

Balanced sample weights give failures more influence during training. Conceptually, the network minimizes weighted binary cross-entropy:

$$L=-\frac{1}{n}\sum_i w_i[y_i\log(\hat p_i)+(1-y_i)\log(1-\hat p_i)]+\alpha\lVert W\rVert_2^2$$

Early stopping retains the model when internal validation performance stops improving. This limits unnecessary training and helps control overfitting.

In [ ]:
training_history = neural_experiment.training_history()
display(training_history.tail())
training_history.to_csv(OUTPUT_DIR / "training_history.csv", index=False)

fig, ax1 = plt.subplots(figsize=(9, 5))
ax1.plot(
    training_history["iteration"],
    training_history["training_loss"],
    color="#1f6f78",
    label="Training loss",
)
ax1.set_xlabel("Training iteration")
ax1.set_ylabel("Weighted training loss")
ax2 = ax1.twinx()
ax2.plot(
    training_history["iteration"],
    training_history["validation_accuracy"],
    color="#bd4f3b",
    alpha=0.8,
    label="Validation accuracy",
)
ax2.set_ylabel("Internal validation accuracy")
fig.suptitle("Neural network learning history")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "neural_network_learning_curve.png", dpi=200, bbox_inches="tight")
plt.show()

## 4. Default and Training-Selected Thresholds

The default threshold is 0.5. A second threshold is selected by maximizing F1 on five-fold out-of-fold predictions from the training set only. The final test labels are not used to choose it.

This separates two questions:

1. How does the network compare at the standard threshold?
2. How does its operating behavior change when the alarm threshold is selected without using the test set?

In [ ]:
selected_threshold_row = neural_experiment.select_threshold_from_training(folds=5)
selected_threshold = neural_experiment.selected_threshold

network_metrics = pd.concat(
    [
        neural_experiment.metrics_frame(threshold=0.5).assign(
            configuration="neural_default_0.5"
        ),
        neural_experiment.metrics_frame(threshold=selected_threshold).assign(
            configuration="neural_training_selected"
        ),
    ],
    ignore_index=True,
)

display(selected_threshold_row.to_frame().T)
display(network_metrics)
network_metrics.to_csv(OUTPUT_DIR / "neural_network_test_metrics.csv", index=False)
neural_experiment.threshold_selection_table.to_csv(
    OUTPUT_DIR / "training_oof_threshold_selection.csv", index=False
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, threshold, title in [
    (axes[0], 0.5, "Default threshold 0.5"),
    (axes[1], selected_threshold, f"Training-selected threshold {selected_threshold:.2f}"),
]:
    matrix = confusion_matrix(
        neural_experiment.split_data.y_test,
        neural_experiment.predict(threshold=threshold),
    )
    sns.heatmap(matrix, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("Actual label")
fig.tight_layout()
fig.savefig(FIGURE_DIR / "neural_network_confusion_matrices.png", dpi=200, bbox_inches="tight")
plt.show()

### Threshold Interpretation

At threshold 0.5, the network detected 61 of 68 failures with precision 0.2711, recall 0.8971, and F1 0.4164. The training-only procedure selected threshold 0.86. On the untouched test set, this raised precision to 0.5281 and F1 to 0.5987, while recall fell to 0.6912 and missed failures increased from 7 to 21. The threshold is therefore a decision policy, not merely a technical constant.

## 5. Compare the Tree and Neural Network

The base and engineered decision trees are reproduced on the same held-out records. The engineered tree is included here so the neural comparison is complete; Notebook 06 derives and interprets its relationship features in detail. PR-AUC is especially useful for this imbalanced dataset because it evaluates probability ranking across precision-recall operating points.

In [ ]:
def add_engineered_features(frame):
    engineered = frame.copy()
    engineered["Temperature difference"] = (
        engineered["Process temperature"] - engineered["Air temperature"]
    )
    engineered["Power proxy"] = engineered["Torque"] * engineered["Rotational speed"]
    engineered["Wear torque interaction"] = engineered["Tool wear"] * engineered["Torque"]
    engineered["Speed torque ratio"] = engineered["Rotational speed"] / (
        engineered["Torque"] + 1e-6
    )
    return engineered


engineered_numeric_features = NUMERIC_FEATURES + [
    "Temperature difference",
    "Power proxy",
    "Wear torque interaction",
    "Speed torque ratio",
]


def build_tree_pipeline(numeric_features):
    return Pipeline(
        [
            (
                "preprocessor",
                ColumnTransformer(
                    [
                        ("numeric", StandardScaler(), numeric_features),
                        (
                            "categorical",
                            OneHotEncoder(handle_unknown="ignore"),
                            CATEGORICAL_FEATURES,
                        ),
                    ]
                ),
            ),
            (
                "classifier",
                DecisionTreeClassifier(
                    max_depth=4,
                    class_weight="balanced",
                    random_state=RANDOM_STATE,
                ),
            ),
        ]
    )


def comparison_row(name, actual, probabilities, threshold):
    predictions = (probabilities >= threshold).astype(int)
    matrix = confusion_matrix(actual, predictions)
    return {
        "configuration": name,
        "threshold": threshold,
        "accuracy": accuracy_score(actual, predictions),
        "precision": precision_score(actual, predictions, zero_division=0),
        "recall": recall_score(actual, predictions, zero_division=0),
        "f1": f1_score(actual, predictions, zero_division=0),
        "pr_auc": average_precision_score(actual, probabilities),
        "roc_auc": roc_auc_score(actual, probabilities),
        "false_positives": int(matrix[0, 1]),
        "false_negatives": int(matrix[1, 0]),
        "true_positives": int(matrix[1, 1]),
    }


split_data = neural_experiment.split_data
base_tree = build_tree_pipeline(NUMERIC_FEATURES)
base_tree.fit(split_data.x_train, split_data.y_train)
base_tree_probabilities = base_tree.predict_proba(split_data.x_test)[:, 1]

engineered_tree = build_tree_pipeline(engineered_numeric_features)
engineered_tree.fit(
    add_engineered_features(split_data.x_train),
    split_data.y_train,
)
engineered_tree_probabilities = engineered_tree.predict_proba(
    add_engineered_features(split_data.x_test)
)[:, 1]

neural_probabilities = neural_experiment.predict_proba()
model_comparison = pd.DataFrame(
    [
        comparison_row("base_tree", split_data.y_test, base_tree_probabilities, 0.5),
        comparison_row(
            "engineered_tree", split_data.y_test, engineered_tree_probabilities, 0.5
        ),
        comparison_row("neural_default", split_data.y_test, neural_probabilities, 0.5),
        comparison_row(
            "neural_training_selected",
            split_data.y_test,
            neural_probabilities,
            selected_threshold,
        ),
    ]
)

display(model_comparison)
model_comparison.to_csv(OUTPUT_DIR / "holdout_model_comparison.csv", index=False)

In [ ]:
plt.figure(figsize=(8, 5.5))
for name, probabilities in [
    ("Base tree", base_tree_probabilities),
    ("Engineered tree", engineered_tree_probabilities),
    ("Neural network", neural_probabilities),
]:
    precision, recall, _ = precision_recall_curve(split_data.y_test, probabilities)
    average_precision = average_precision_score(split_data.y_test, probabilities)
    plt.plot(recall, precision, label=f"{name} (AP={average_precision:.3f})")
plt.axhline(split_data.y_test.mean(), color="gray", linestyle="--", label="Failure prevalence")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-recall comparison")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURE_DIR / "model_precision_recall_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

### Fair Comparison Rule

At the common 0.5 threshold, the neural network did not outperform either tree on F1. Its PR-AUC of 0.6185 exceeded the base tree's 0.5246 but remained below the engineered tree's 0.7918. Its ROC-AUC of 0.9618 was the highest, showing strong overall ranking even though its chosen 0.5 operating point created 164 false positives. The `neural_training_selected` row is a separate operating-policy experiment and must not be presented as though architecture alone produced its F1 improvement.

## 6. Global Neural-Network Explanation

A neural network has no short decision path or impurity importance. Permutation importance measures how much test PR-AUC falls when one original feature is shuffled. This is model-agnostic and preserves `Type` as one complete feature.

In [ ]:
permutation_result = permutation_importance(
    neural_experiment.model,
    split_data.x_test,
    split_data.y_test,
    scoring="average_precision",
    n_repeats=20,
    random_state=RANDOM_STATE,
)
permutation_table = pd.DataFrame(
    {
        "feature": INPUT_FEATURES,
        "mean_pr_auc_decrease": permutation_result.importances_mean,
        "std_pr_auc_decrease": permutation_result.importances_std,
    }
).sort_values("mean_pr_auc_decrease", ascending=False)

display(permutation_table)
permutation_table.to_csv(OUTPUT_DIR / "neural_permutation_importance.csv", index=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=permutation_table, x="mean_pr_auc_decrease", y="feature", color="#287271")
plt.xlabel("Mean decrease in test PR-AUC")
plt.ylabel("")
plt.title("Neural-network permutation importance")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "neural_permutation_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 7. Model-Agnostic SHAP for the Neural Network

Permutation importance ranked torque, air temperature, rotational speed, process temperature, and tool wear as influential; `Type` caused almost no mean PR-AUC decrease. Permutation SHAP treats the trained network as a prediction function. `Type` is encoded only for the explainer and converted back before the pipeline predicts. This avoids independently perturbing impossible one-hot combinations.

Unlike Tree SHAP, this model-agnostic method is approximate and computationally more expensive.

In [ ]:
type_to_code = {"L": 0, "M": 1, "H": 2}
code_to_type = {value: key for key, value in type_to_code.items()}


def encode_raw_features(frame):
    encoded = frame[INPUT_FEATURES].copy()
    encoded["Type"] = encoded["Type"].map(type_to_code)
    return encoded.astype(float).to_numpy()


def encoded_failure_probability(values):
    frame = pd.DataFrame(np.asarray(values), columns=INPUT_FEATURES)
    type_codes = np.clip(np.rint(frame["Type"]), 0, 2).astype(int)
    frame["Type"] = type_codes.map(code_to_type)
    return neural_experiment.model.predict_proba(frame)[:, 1]


x_train_encoded = encode_raw_features(split_data.x_train)
x_test_encoded = encode_raw_features(split_data.x_test)
background = shap.sample(x_train_encoded, 50, random_state=RANDOM_STATE)
neural_shap_explainer = shap.Explainer(
    encoded_failure_probability,
    background,
    algorithm="permutation",
    feature_names=INPUT_FEATURES,
    seed=RANDOM_STATE,
)

rng = np.random.default_rng(RANDOM_STATE)
global_positions = np.sort(rng.choice(len(x_test_encoded), size=200, replace=False))
neural_shap_values = neural_shap_explainer(
    x_test_encoded[global_positions],
    max_evals=2 * len(INPUT_FEATURES) + 1,
)
neural_shap_importance = pd.DataFrame(
    {
        "feature": INPUT_FEATURES,
        "mean_absolute_shap": np.abs(neural_shap_values.values).mean(axis=0),
    }
).sort_values("mean_absolute_shap", ascending=False)

display(neural_shap_importance)
neural_shap_importance.to_csv(OUTPUT_DIR / "neural_shap_global_importance.csv", index=False)

plt.figure(figsize=(8, 4.5))
sns.barplot(data=neural_shap_importance, x="mean_absolute_shap", y="feature", color="#7c5b3e")
plt.xlabel("Mean absolute SHAP value")
plt.ylabel("")
plt.title("Approximate global SHAP importance for neural network")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "neural_shap_global_importance.png", dpi=200, bbox_inches="tight")
plt.show()

## 8. Local Neural-Network Explanations

One TP, FP, FN, and TN from the default threshold are explained with both SHAP and LIME. LIME operates in original sensor units and treats `Type` as categorical.

In [ ]:
actual_values = split_data.y_test.to_numpy()
default_predictions = neural_experiment.predict(threshold=0.5)
case_masks = {
    "true_positive": (actual_values == 1) & (default_predictions == 1),
    "false_positive": (actual_values == 0) & (default_predictions == 1),
    "false_negative": (actual_values == 1) & (default_predictions == 0),
    "true_negative": (actual_values == 0) & (default_predictions == 0),
}
case_positions = {
    name: int(np.flatnonzero(mask)[0]) for name, mask in case_masks.items()
}

local_values = neural_shap_explainer(
    x_test_encoded[list(case_positions.values())],
    max_evals=2 * len(INPUT_FEATURES) + 1,
)
local_summary_rows = []
for local_index, (case_name, row_position) in enumerate(case_positions.items()):
    local_table = pd.DataFrame(
        {
            "feature": INPUT_FEATURES,
            "raw_feature_value": split_data.x_test.iloc[row_position][INPUT_FEATURES].values,
            "shap_value_for_failure": local_values.values[local_index],
        }
    )
    local_table["absolute_shap_value"] = local_table["shap_value_for_failure"].abs()
    local_table = local_table.sort_values("absolute_shap_value", ascending=False)
    local_table.to_csv(OUTPUT_DIR / f"neural_shap_local_{case_name}.csv", index=False)
    local_summary_rows.append(
        {
            "case_type": case_name,
            "test_position": row_position,
            "actual": actual_values[row_position],
            "predicted": default_predictions[row_position],
            "failure_probability": neural_probabilities[row_position],
            "top_shap_feature": local_table.iloc[0]["feature"],
            "top_shap_value": local_table.iloc[0]["shap_value_for_failure"],
        }
    )

local_shap_summary = pd.DataFrame(local_summary_rows)
display(local_shap_summary)
local_shap_summary.to_csv(OUTPUT_DIR / "neural_local_case_summary.csv", index=False)

reconstructed = local_values.base_values + local_values.values.sum(axis=1)
selected_probabilities = neural_probabilities[list(case_positions.values())]
print(
    "Maximum local SHAP reconstruction error:",
    np.abs(reconstructed - selected_probabilities).max(),
)

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=x_train_encoded,
    feature_names=INPUT_FEATURES,
    class_names=["No failure", "Failure"],
    categorical_features=[0],
    categorical_names={0: ["L", "M", "H"]},
    mode="classification",
    random_state=RANDOM_STATE,
)


def encoded_predict_proba(values):
    failure_probability = encoded_failure_probability(values)
    return np.column_stack([1 - failure_probability, failure_probability])


lime_summary_rows = []
for case_name, row_position in case_positions.items():
    explanation = lime_explainer.explain_instance(
        x_test_encoded[row_position],
        encoded_predict_proba,
        labels=[1],
        num_features=len(INPUT_FEATURES),
        num_samples=3000,
    )
    lime_table = pd.DataFrame(
        explanation.as_list(label=1),
        columns=["lime_condition", "lime_weight_for_failure"],
    )
    lime_table.to_csv(OUTPUT_DIR / f"neural_lime_local_{case_name}.csv", index=False)
    lime_summary_rows.append(
        {
            "case_type": case_name,
            "surrogate_r2": explanation.score,
            "top_lime_condition": lime_table.iloc[0]["lime_condition"],
            "top_lime_weight": lime_table.iloc[0]["lime_weight_for_failure"],
        }
    )

lime_summary = pd.DataFrame(lime_summary_rows)
display(lime_summary)
lime_summary.to_csv(OUTPUT_DIR / "neural_lime_case_summary.csv", index=False)

### Explanation Interpretation

SHAP reconstructed the four selected network probabilities with maximum numerical error $5.55\times10^{-16}$. The selected true positive was mainly pushed toward failure by air temperature, while the false positive was dominated by torque. In the false negative, torque pushed toward failure but process temperature and other features pushed more strongly in the opposite direction, leaving probability 0.0134. LIME surrogate $R^2$ ranged from 0.1164 to 0.3557, so these readable local approximations remain incomplete. Neither SHAP nor LIME proves physical causality.

## 9. Five-Fold Model Validation

The final comparison evaluates the base tree, engineered tree, and neural network across the same five stratified folds. Network sample weights are recalculated from the dataset and passed into each training fold.

In [ ]:
x_base = data[INPUT_FEATURES].copy()
x_engineered = add_engineered_features(x_base)
y = data["Machine failure"].copy()
cross_validation = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
}

cv_rows = []
model_specs = [
    ("base_tree", build_tree_pipeline(NUMERIC_FEATURES), x_base, None),
    (
        "engineered_tree",
        build_tree_pipeline(engineered_numeric_features),
        x_engineered,
        None,
    ),
    (
        "neural_network",
        neural_experiment.build_pipeline(),
        x_base,
        {"classifier__sample_weight": compute_sample_weight("balanced", y)},
    ),
]

for model_name, model, feature_data, fit_parameters in model_specs:
    arguments = {
        "estimator": model,
        "X": feature_data,
        "y": y,
        "cv": cross_validation,
        "scoring": scoring,
    }
    if fit_parameters is not None:
        arguments["params"] = fit_parameters
    scores = cross_validate(**arguments)
    for fold in range(5):
        cv_rows.append(
            {
                "model": model_name,
                "fold": fold + 1,
                **{
                    metric: scores[f"test_{metric}"][fold]
                    for metric in scoring
                },
            }
        )

cv_results = pd.DataFrame(cv_rows)
cv_summary = cv_results.groupby("model")[list(scoring)].agg(["mean", "std"])
display(cv_summary.round(4))
cv_results.to_csv(OUTPUT_DIR / "five_fold_model_results.csv", index=False)
cv_summary.to_csv(OUTPUT_DIR / "five_fold_model_summary.csv")

In [ ]:
cv_long = cv_results.melt(
    id_vars=["model", "fold"],
    value_vars=["precision", "recall", "f1", "pr_auc"],
    var_name="metric",
    value_name="score",
)
plt.figure(figsize=(10, 5.5))
sns.boxplot(data=cv_long, x="metric", y="score", hue="model")
plt.ylim(0, 1)
plt.title("Five-fold comparison: tree models and neural network")
plt.tight_layout()
plt.savefig(FIGURE_DIR / "five_fold_neural_comparison.png", dpi=200, bbox_inches="tight")
plt.show()

## 10. Responsible AI Interpretation

Across five folds, the neural network achieved mean precision 0.2279, recall 0.8525, F1 0.3593, PR-AUC 0.5648, and ROC-AUC 0.9501. It ranked failures smoothly and achieved the highest mean ROC-AUC, but the engineered tree remained stronger on mean precision, recall, F1, and PR-AUC. The neural network therefore adds nonlinear modeling capacity but reduces intrinsic interpretability without becoming the preferred primary model.

Its inclusion is justified as a transparent comparison:

- Report when it does not outperform the tree at the same threshold.
- Separate ranking quality from threshold-dependent classification metrics.
- Select any alternative threshold without using final test labels.
- Evaluate post-hoc explanations rather than presenting them as guaranteed truth.
- Compare predictive gains with explanation complexity and missed-failure consequences.

For this synthetic tabular dataset, a more complex model is not automatically a more responsible model.

## 11. Saved Evidence

Tables and presentation-ready figures are saved under `outputs/neural_network_comparison/`. The key figures are:

- `neural_network_learning_curve.png`
- `neural_network_confusion_matrices.png`
- `model_precision_recall_comparison.png`
- `neural_permutation_importance.png`
- `neural_shap_global_importance.png`
- `five_fold_neural_comparison.png`